# Opus-MT + LoRA: Classical Chinese Poetry → English

Fine-tunes `Helsinki-NLP/opus-mt-zh-en` (MarianMT, ~74M params) on the PoetMT poetry dataset using LoRA.

**Why opus-mt over mT5-base?** opus-mt-zh-en is already pre-trained on OPUS Chinese→English data, so it starts as a working translator. mT5-base was only pre-trained on unsupervised text and cannot translate without extensive fine-tuning first.

**Estimated training time (15 epochs):**
- Free Colab T4: ~20–35 min
- Local GPU (RTX 3060-class): ~50–70 min

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers peft accelerate sentencepiece sacrebleu evaluate datasets bert_score
!pip install -q "torchao>=0.16.0"


In [ ]:
# ── Cell 2: Mount Google Drive (adapter will be saved here) ───────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Clone repo (private) ─────────────────────────────────────────
# Colab sidebar → Secrets → add GITHUB_TOKEN (fine-grained PAT, contents read)
import os
from google.colab import userdata
from getpass import getpass

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    GITHUB_TOKEN = getpass("Paste GitHub PAT: ")

REPO_DIR = "/content/chinese_poetry_translation"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/emmah-3815/chinese_poetry_translation.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout juqy-dev
!git pull
print("Working dir:", os.getcwd())
!git log --oneline -3


In [ ]:
# ── Cell 4: Download raw data (PoetMT + CCPM) ────────────────────────────
%cd /content/chinese_poetry_translation
import os

# Clone PoetMT into data/PoetMT-main/PoetMT-main so all_poems/ is at the expected path
if not os.path.exists("data/PoetMT-main/PoetMT-main/all_poems/tang.jsonl"):
    os.makedirs("data/PoetMT-main", exist_ok=True)
    !git clone https://github.com/andongBlue/PoetMT.git data/PoetMT-main/PoetMT-main
    print("PoetMT contents:", os.listdir("data/PoetMT-main/PoetMT-main"))
else:
    print("PoetMT already present")

# Clone CCPM directly into data/CCPM-master
if not os.path.exists("data/CCPM-master/train.jsonl"):
    !git clone https://github.com/THUNLP-AIPoet/CCPM.git data/CCPM-master
    print("CCPM contents:", os.listdir("data/CCPM-master"))
else:
    print("CCPM already present")


In [ ]:
# ── Cell 5: Build dataset ─────────────────────────────────────────────────
%cd /content/chinese_poetry_translation
!python build_dataset.py


In [ ]:
# ── Cell 6: Smoke test (1 epoch — verify pipeline before full run) ────────
%cd /content/chinese_poetry_translation
import json as _json
from collections import Counter
_tasks = Counter(_json.loads(l).get("task") for l in open("data/combined/train.jsonl") if l.strip())
print("Dataset task counts:", dict(_tasks))
assert _tasks.get("translation", 0) > 0, "No translation records found — re-run Cell 5"
!python pipelines/opus_mt/train_opus_mt.py --data_dir data/combined --output_dir /tmp/opus-mt-smoke --epochs 1 --precision bf16
print("Smoke test done — check output above before running Cell 7")


In [ ]:
# ── Cell 7: Full training (15 epochs, saved to Google Drive) ─────────────
%cd /content/chinese_poetry_translation
OUTPUT_DIR = "/content/drive/MyDrive/models/opus-mt-poetry"
!python pipelines/opus_mt/train_opus_mt.py --data_dir data/combined --output_dir {OUTPUT_DIR} --epochs 15 --precision bf16


In [ ]:
# ── Cell 8: Evaluate trained adapter on canonical 78-poem test set ───────
%cd /content/chinese_poetry_translation
OUTPUT_DIR  = "/content/drive/MyDrive/models/opus-mt-poetry"
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
EVAL_OUT    = f"{OUTPUT_DIR}/eval_results"
!python eval_e2_mt5.py --adapter_dir {ADAPTER_DIR} --base_model Helsinki-NLP/opus-mt-zh-en --no_task_prefix --flat_test --output_dir {EVAL_OUT}


In [ ]:
# ── Cell 9: Baseline eval (no adapter, SAME decoding) ────────────────────
# Apples-to-apples reference for Cell 8: identical anti-repeat decoding,
# no LoRA. The "did fine-tuning help" delta = Cell 8 BERTScore-F - this.
%cd /content/chinese_poetry_translation
BASE_EVAL_OUT = "/content/drive/MyDrive/models/opus-mt-baseline/eval_results"
!python eval_e2_mt5.py --baseline --base_model Helsinki-NLP/opus-mt-zh-en --no_task_prefix --flat_test --output_dir {BASE_EVAL_OUT}


---
## mT5-base (580M) — size-controlled row for Table 1

mT5-base is the 0.58B encoder-decoder point (vs Qwen-0.5B-base in the paper). Two rows: **LoRA** and **Base**.

⚠️ **The old `e2-mt5-fp32-v2` adapter is leakage-contaminated** — it was trained *before* the canonical-aware split fix, so ~55 of the 78 canonical test poems were in its training set. Evaluating it on the canonical 78 would be ~70 % memorization and is **invalid for the paper**. Cell 11 retrains mT5 on the clean `data/combined` split.

The **Base** row (Cell 12) needs no retrain — a non-fine-tuned model can't be contaminated.

**Use an A100 runtime for Cell 11** (mT5-base is 580M; T4 is very slow). bf16 is auto-enabled on A100.

In [ ]:
# ── Cell 11: Retrain mT5-base on CLEAN data (size-controlled LoRA row + train VRAM) ──
# REQUIRES A100 runtime (Runtime → Change runtime type → A100 GPU). bf16 auto-used.
# Writes train_stats.json (train_peak_vram_gb) to MT5_OUT.
%cd /content/chinese_poetry_translation
MT5_OUT = "/content/drive/MyDrive/models/e2-mt5-clean"
!python train_e2_mt5.py --data_dir data/combined --output_dir {MT5_OUT} --epochs 15 --early_stopping_patience 4 --precision bf16


In [ ]:
# ── Cell 12: mT5-base BASELINE eval on canonical 78 (no adapter) ─────────
# Non-fine-tuned → no leakage concern. mT5 uses the task prefix (no --no_task_prefix).
%cd /content/chinese_poetry_translation
!python eval_e2_mt5.py --baseline --flat_test --output_dir /content/drive/MyDrive/models/mt5-base-baseline/eval_results


In [ ]:
# ── Cell 13: mT5-base + LoRA eval on canonical 78 (clean adapter from Cell 11) ──
# mT5 USES the task prefix (do NOT pass --no_task_prefix). Same anti-repeat decoding as opus-mt.
%cd /content/chinese_poetry_translation
MT5_OUT = "/content/drive/MyDrive/models/e2-mt5-clean"
!python eval_e2_mt5.py --adapter_dir {MT5_OUT}/lora_adapter --flat_test --output_dir {MT5_OUT}/eval_results
